In [45]:
import re
import numpy as np
import pandas as pd

## Import Beagle sgRNA annotation


In [40]:
variants = pd.read_csv(f"../data/BeagleCoelho/EG_collab-guides.txt", sep="\t")
variants.shape

(24752, 24)

In [46]:
variants.sample(10)

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Taxon,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,...,PAM Sequence,sgRNA Sequence Start Pos. (global),sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note
475,ENST00000262948.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,TG,4097335,sense,4097351G>A,C_4,920-8C>T,(NC),Intron,NaN,NaN
7831,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,AG,55181285,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6244,ENST00000275493.7,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,TG,55156636,antisense,55156652T>C,A_4,1126T>C,Phe376Leu,Missense,NaN,NaN
5832,ENST00000275493.7,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,GG,55154053,antisense,"55154066T>C, 55154069T>C","A_7, A_4","803T>C, 806T>C","Met268Thr, Leu269Pro","Missense, Missense",NaN,NaN
9257,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,AG,55205352,sense,"55205356C>T, 55205359C>T","C_5, C_8","3372C>T, 3375C>T","His1124His, Tyr1125Tyr","Silent, Silent",NaN,NaN
3426,ENST00000263967.4,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000003.12,ENSG00000121879,PIK3CA,+,...,AG,179219238,antisense,179219251T>C,A_7,1720T>C,Trp574Arg,Missense,NaN,NaN
21952,ENST00000646891.2,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000007.14,ENSG00000157764,BRAF,-,...,GG,140834629,antisense,140834635A>G;140834634A>G,A_7;A_6,478T>C;479T>C,Phe160Pro,Missense,NaN,NaN
10751,ENST00000307102.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000015.10,ENSG00000169032,MAP2K1,+,...,TG,66487217,sense,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12753,ENST00000366794.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,TG,226361999,sense,226362014G>A;226362013G>A,C_5;C_6,2918C>T;2919C>T,Thr973Ile,Missense,NaN,NaN
5325,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,GG,55146601,antisense,"55146613G>A, 55146617G>A","C_8, C_4","432G>A, 436G>A","Leu144Leu, Gly146Ser","Silent, Missense",NaN,NaN


In [49]:
variants.loc[5325]

Input                                              ENST00000275493.7
CRISPR Enzyme                                             SpyoCas9NG
Edit Type                                                        C-T
Edit Window                                                     4..8
Target Taxon                                                    9606
Target Assembly                                               GRCh38
Target Genome Sequence                                  NC_000007.14
Target Gene ID                                       ENSG00000146648
Target Gene Symbol                                              EGFR
Target Gene Strand                                                 +
Target Transcript ID                               ENST00000275493.7
Target Domain                                                    CDS
sgRNA Sequence                                  CGCCATGCAGGATTTCTGCG
sgRNA Context Sequence                ACGGCGCCATGCAGGATTTCTGCGGGAGAA
PAM Sequence                      

In [57]:
variants.loc[21952]

Input                                              ENST00000646891.2
CRISPR Enzyme                                             SpyoCas9NG
Edit Type                                                        A-G
Edit Window                                                     4..8
Target Taxon                                                    9606
Target Assembly                                               GRCh38
Target Genome Sequence                                  NC_000007.14
Target Gene ID                                       ENSG00000157764
Target Gene Symbol                                              BRAF
Target Gene Strand                                                 -
Target Transcript ID                               ENST00000646891.2
Target Domain                                                    CDS
sgRNA Sequence                                  GCAGGAAGACTCTAACGATA
sgRNA Context Sequence                TTGGGCAGGAAGACTCTAACGATAGGTTTT
PAM Sequence                      

In [56]:
variants["sgRNA Sequence"].value_counts().value_counts()

count
2    12346
4       15
Name: count, dtype: int64

In [53]:
variants.query("`sgRNA Sequence` == 'GGAGTTGGCCATGGAGTCGA'")

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Taxon,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,...,PAM Sequence,sgRNA Sequence Start Pos. (global),sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note
890,ENST00000262948.10,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,TG,4101046,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN
891,ENST00000262948.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,TG,4101046,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10410,ENST00000307102.10,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000015.10,ENSG00000169032,MAP2K1,+,...,TG,66481833,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10411,ENST00000307102.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000015.10,ENSG00000169032,MAP2K1,+,...,TG,66481833,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Format for ENSEMBL VEP input

Use the Beagle columns [input, Nucleotide Edits] and format as HGVS identifiers for ENSEMBL VEP. Examples include:

- `ENST00000618231.3:c.9G>C`
- `ENST00000471631.1:c.28_33delTCGCGG`
- `ENST00000285667.3:c.1047_1048insC`
- `5:g.140532G>C`


In [47]:
def format_variant(variant="1033T>C;1034T>C;1035T>C") -> str:
    if ";" not in variant:
        return variant

    else:
        # input: 1033T>C;1034T>C;1035T>C
        # output: 1033_1035T>C

        # Parse the input string into a list of tuples (position, original_base, altered_base)
        edits = re.findall(r"(\d+)([A-Z])>([A-Z])", variant)

        # Extract positions from the parsed data
        positions = [int(edit[0]) for edit in edits]

        # Check if the positions are consecutive
        if all(positions[i] + 1 == positions[i + 1] for i in range(len(positions) - 1)):
            # Return the compact format
            return f"{positions[0]}_{positions[-1]}{edits[0][1]}>{edits[0][2]}"

        else:
            # If not consecutive, return the input as it is
            return variant

'1033_1035T>C'

In [48]:
ensembl_vep_input = "\n".join(
    [
        f"{g}:c.{format_variant(e)}"
        for g, v in variants[["Input", "Nucleotide Edits"]].values
        if v is not np.nan
        for e in v.split(", ")
    ]
)
print(ensembl_vep_input)

ENST00000262948.10:c.*12C>T
ENST00000262948.10:c.*14C>T
ENST00000262948.10:c.*7C>T
ENST00000262948.10:c.*8C>T
ENST00000262948.10:c.*2A>G
ENST00000262948.10:c.1203A>G
ENST00000262948.10:c.*2A>G
ENST00000262948.10:c.*1C>T
ENST00000262948.10:c.*9G>A
ENST00000262948.10:c.*10G>A
ENST00000262948.10:c.*11G>A
ENST00000262948.10:c.*9G>A
ENST00000262948.10:c.*10G>A
ENST00000262948.10:c.*11G>A
ENST00000262948.10:c.*5G>A
ENST00000262948.10:c.*6G>A
ENST00000262948.10:c.*9G>A
ENST00000262948.10:c.*4T>C
ENST00000262948.10:c.*5G>A
ENST00000262948.10:c.*6G>A
ENST00000262948.10:c.1196_1197C>T
ENST00000262948.10:c.*4T>C
ENST00000262948.10:c.*3G>A
ENST00000262948.10:c.*5G>A
ENST00000262948.10:c.*6G>A
ENST00000262948.10:c.1196_1197C>T
ENST00000262948.10:c.1194C>T
ENST00000262948.10:c.1196_1197C>T
ENST00000262948.10:c.*4T>C
ENST00000262948.10:c.*3G>A
ENST00000262948.10:c.1201T>C
ENST00000262948.10:c.1202G>A
ENST00000262948.10:c.1192A>G
ENST00000262948.10:c.1191C>T
ENST00000262948.10:c.1193_1194C>T
ENST00000

In [41]:
with open("../data/BeagleCoelho/EG_collab-guides-VEP-HGVS-input.txt", "w") as f:
    f.write(ensembl_vep_input)

## ENSEMBL VEP command


In [ ]:
!./vep --af --appris --biotype --buffer_size 500 --check_existing --distance 5000 --domains --gencode_primary --mane --plugin SpliceAI,snv=[path_to]/spliceai_scores.masked.snv.hg38.vcf.gz,indel=[path_to]/spliceai_scores.masked.indel.hg38.vcf.gz,snv_ensembl=[path_to]/spliceai_scores.raw.snv.ensembl_mane.grch38.110.vcf.gz --plugin AncestralAllele,[path_to]/homo_sapiens_ancestor_GRCh38_109.fa.gz --plugin Enformer,file=[path_to]/enformer_grch38.vcf.gz --plugin AlphaMissense,file=[path_to]/AlphaMissense_hg38.tsv.gz --plugin dbscSNV,[path_to]/dbscSNV1.1_GRCh38.txt.gz --plugin ClinPred,file=[path_to]/ClinPred_hg38_sorted_tabbed.tsv.gz --plugin MaxEntScan,[path_to]/maxentscan --plugin LOEUF,file=[path_to]/loeuf_dataset_grch38.tsv.gz,match_by=gene --plugin REVEL,file=[path_to]/new_tabbed_revel_grch38.tsv.gz --plugin CADD,snv=[path_to]/CADD_GRCh38_1.7_whole_genome_SNVs.tsv.gz,indels=[path_to]/CADD_GRCh38_1.7_InDels.tsv.gz --plugin EVE,file=[path_to]/eve_merged.vcf.gz --plugin mutfunc,db=[path_to]/mutfunc_data.db,motif=1,int=1,mod=1,exp=1 --plugin Blosum62 --plugin NMD --plugin MaveDB,file=[path_to]/MaveDB_variants.tsv.gz --polyphen b --pubmed --regulatory --show_ref_allele --sift b --species homo_sapiens --symbol --transcript_version --tsl --uploaded_allele --cache --input_file [input_data] --output_file [output_file]